# Setup and Dataset Loading

In [ ]:
!pip install GEOparse -q

import GEOparse
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

gse = GEOparse.get_GEO(geo="GSE6613", destdir="./")
df = gse.pivot_samples('VALUE')

04-May-2026 15:29:36 DEBUG utils - Directory ./ already exists. Skipping.
DEBUG:GEOparse:Directory ./ already exists. Skipping.
04-May-2026 15:29:36 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE6nnn/GSE6613/soft/GSE6613_family.soft.gz to ./GSE6613_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE6nnn/GSE6613/soft/GSE6613_family.soft.gz to ./GSE6613_family.soft.gz
100%|██████████| 16.1M/16.1M [00:01<00:00, 16.5MB/s]
04-May-2026 15:29:37 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
04-May-2026 15:29:37 DEBUG downloader - Moving /tmp/tmppiiwxqnm to /content/GSE6613_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmppiiwxqnm to /content/GSE6613_family.soft.gz
04-May-2026 15:29:37 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE6nnn/GSE6613/soft/GSE6613_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE6nnn/GSE6613/soft/GSE661

# Data Preprocessing

In [ ]:
X = df.T
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Principal Component Analysis (PCA)

In [ ]:
pca_95 = PCA(n_components=0.95, random_state=42)
pca_95.fit(X_scaled)

PCA(n_components=0.95, random_state=42)

# Gene Selection Function

In [ ]:
def get_top_genes(pca_model, feature_names, num_pcs, top_n=20):
    loadings = pca_model.components_[:num_pcs]
    explained_variance = pca_model.explained_variance_ratio_[:num_pcs]
    abs_loadings = np.abs(loadings)

    ultimate_weights = np.dot(explained_variance, abs_loadings)

    gene_weights_df = pd.DataFrame({
        'Gene': feature_names,
        'Ultimate_Weight': ultimate_weights
    })

    gene_weights_df = gene_weights_df.sort_values(by='Ultimate_Weight', ascending=False)

    return gene_weights_df.head(top_n)

gene_names = X.columns

# Top 20 Genes Using Top 5 PCs

In [ ]:
top_20_genes_5_pcs = get_top_genes(pca_95, gene_names, 5)

print("Top 20 Genes (Using Top 5 PCs):")
display(top_20_genes_5_pcs)

Top 20 Genes (Using Top 5 PCs):


,Gene,Ultimate_Weight
439,200912_s_at,0.002783
3546,204020_at,0.002721
22273,AFFX-r2-Ec-bioD-3_at,0.002690
154,200627_at,0.002689
83,200063_s_at,0.002673
578,201051_at,0.002657
20561,221199_at,0.002645
385,200858_s_at,0.002614
11099,211699_x_at,0.002611
16598,217232_x_at,0.002607


# Top 20 Genes Using Top 10 PCs

In [ ]:
top_20_genes_10_pcs = get_top_genes(pca_95, gene_names, 10)

print("Top 20 Genes (Using Top 10 PCs):")
display(top_20_genes_10_pcs)

Top 20 Genes (Using Top 10 PCs):


,Gene,Ultimate_Weight
10606,211165_x_at,0.003287
20561,221199_at,0.003282
15105,215733_x_at,0.003234
15780,216410_at,0.003232
9648,210165_at,0.003224
5953,206428_s_at,0.003221
10856,211448_s_at,0.003218
11302,211915_s_at,0.003215
19015,219652_s_at,0.003180
19369,220006_at,0.003174


# Overlap Analysis

In [ ]:
genes_5_pcs = set(top_20_genes_5_pcs['Gene'])
genes_10_pcs = set(top_20_genes_10_pcs['Gene'])

only_in_5 = genes_5_pcs - genes_10_pcs
only_in_10 = genes_10_pcs - genes_5_pcs
in_both = genes_5_pcs.intersection(genes_10_pcs)

print("Genes only in Top 20 (5 PCs):")
print(only_in_5)
print("\nGenes only in Top 20 (10 PCs):")
print(only_in_10)
print(f"\nNumber of overlapping genes: {len(in_both)}")

Genes only in Top 20 (5 PCs):
{'211745_x_at', '209458_x_at', '201051_at', '200627_at', '204020_at', 'AFFX-r2-Ec-bioD-3_at', '209116_x_at', 'AFFX-r2-Ec-bioD-5_at', '200858_s_at', '217232_x_at', 'AFFX-BioC-3_at', '211699_x_at', '209057_x_at', 'AFFX-r2-Ec-bioC-5_at', '222012_at', '200912_s_at'}

Genes only in Top 20 (10 PCs):
{'219652_s_at', '200006_at', '220006_at', '216346_at', '216410_at', '213828_x_at', '210165_at', '211915_s_at', '216707_at', '205613_at', '211448_s_at', '216926_s_at', '206428_s_at', '215733_x_at', '209441_at', '211165_x_at'}

Number of overlapping genes: 4


# Analysis: Comparing Top 5 PCs vs. Top 10 PCs

**Are there differences between the two lists?**

Yes, there are significant differences between the two lists. When comparing the Top 20 genes selected using 5 Principal Components versus 10 Principal Components, only 4 genes overlap (staying in the Top 20 across both methods). This means 16 genes dropped out of the top rankings, and 16 entirely new genes entered the Top 20 when the additional 5 PCs were included.

**Why is that happening?**

Each Principal Component captures a different pattern of variance within the dataset. The first 5 PCs capture the most dominant, global patterns of variation across the samples. PCs 6 through 10 capture smaller, secondary patterns of variance. Because the "Ultimate Weight" is calculated by summing the absolute loadings multiplied by the explained variance, including PCs 6-10 incorporates genes that are highly influential in those secondary patterns. Even though the explained variance ratio drops for these later PCs, their coefficient loadings accumulate to alter the final overall rankings.

**Comment on the suitability of this approach for gene selection:**

Using PCA loadings is a mathematically sound, unsupervised approach for finding genes that drive the overall variance in a dataset. However, in a biological context, maximum variance does not always equate to disease relevance. A gene might vary widely across samples due to noise, age, or batch effects rather than the specific condition being studied (e.g., Parkinson's vs. Control). Because PCA is unsupervised and does not use the sample labels, it optimizes for overall data spread rather than class separation. Therefore, supervised methods (such as Differential Expression analysis) are generally more suitable for targeted gene selection.